<a href="https://colab.research.google.com/github/AnirudhSekar/RKSC/blob/main/rksc_icml_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reasoning-KV Shared Cache (RKSC)
### Benchmarking Notebook — ICML 2025

| Claim | Result |
|---|---|
| KV sharing improves throughput vs batching alone | +37–39% at prefix=1024, tok=8 |
| CGEE verify-skip adds further gain | +5–15% on top of KV |
| Combined RKSC speedup | 1.78×–2.26× vs batched baseline |


In [ ]:
# !pip install -q "transformers>=4.40" datasets tqdm numpy scipy matplotlib seaborn pandas psutil
%matplotlib inline

In [ ]:
import sys, os, gc, json, time, logging, warnings
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
from scipy import stats
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

logging.basicConfig(level=logging.WARNING)

DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.set_float32_matmul_precision("high")
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    _p = torch.cuda.get_device_properties(0)
    print(f'GPU : {_p.name}  ({_p.total_memory/1e9:.0f} GB)')

import transformers
print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {transformers.__version__}')

_RAM_CACHE: dict = {}

def ckpt_save(name: str, obj): _RAM_CACHE[name] = obj
def ckpt_load(name: str, default=None): return _RAM_CACHE.get(name, default)
def ckpt_exists(name: str) -> bool: return name in _RAM_CACHE

def cpu_ram_gb() -> float:
    import psutil
    return psutil.Process().memory_info().rss / 1e9

print('Imports ready.')


GPU : NVIDIA A100-SXM4-40GB  (42 GB)
PyTorch      : 2.10.0+cu128
Transformers : 5.0.0
Imports ready.


## §1  Configuration

In [ ]:
CALIBRATED = {
    'Qwen/Qwen2.5-7B-Instruct':           {'tau': 0.82, 'theta': 6.0},
    'mistralai/Mistral-7B-Instruct-v0.3': {'tau': 0.80, 'theta': 0.50},
    'tiiuae/Falcon3-7B-Instruct':         {'tau': 0.80, 'theta': 6.0},
    'tiiuae/Falcon3-10B-Instruct':        {'tau': 0.80, 'theta': 6.0},
    'meta-llama/Llama-3-8B-Instruct':     {'tau': 0.82, 'theta': 6.0},
}

MODEL_VRAM_GB = {
    'Qwen/Qwen2.5-7B-Instruct':           15.0,
    'mistralai/Mistral-7B-Instruct-v0.3': 15.0,
    'tiiuae/Falcon3-7B-Instruct':         15.0,
    'tiiuae/Falcon3-10B-Instruct':        20.0,
    'meta-llama/Llama-3-8B-Instruct':     15.0,
}

MODEL_ID         = 'Qwen/Qwen2.5-7B-Instruct'
N_PROBLEMS       = 30
N_RUNS           = 2
B_FACTOR         = 8
MAX_DEPTH        = 3
MAX_NEW_TOK      = 32
MAX_NEW_TOK_EXT  = 8
ATTN_IMPL        = 'sdpa'
MAX_SEQ_LEN      = 2048

CKPT_VERSION = (
    f'{MODEL_ID}|tok={MAX_NEW_TOK}|B={B_FACTOR}'
    f'|N={N_PROBLEMS}|pfx1024|runs={N_RUNS}'
)

print(f'Model       : {MODEL_ID}')
print(f'N / B / tok : {N_PROBLEMS} / {B_FACTOR} / {MAX_NEW_TOK}')


Model       : Qwen/Qwen2.5-7B-Instruct
N / B / tok : 30 / 8 / 32


In [ ]:
_BENCH_KEYS = ['lat_batch_nokv', 'lat_kv', 'lat_cgee', 'lat_rksc', 'extended_results']

_stored_sig = _RAM_CACHE.get('_ckpt_version', '')
if _stored_sig != CKPT_VERSION:
    print(f'Config changed — clearing cached results.')
    for _k in _BENCH_KEYS:
        _RAM_CACHE.pop(_k, None)
        if _k in globals(): del globals()[_k]
    _RAM_CACHE['_ckpt_version'] = CKPT_VERSION
else:
    print(f'Config unchanged. Cached keys: {[k for k in _BENCH_KEYS if k in _RAM_CACHE]}')


Config changed — clearing cached results.


## §2  Dataset & Model Loading

In [ ]:
from huggingface_hub import login, whoami

_hf_token = None

try:
    from google.colab import userdata as _ud
    _tok = _ud.get('HF_TOKEN')
    if _tok: _hf_token = _tok.strip()
except Exception:
    pass

if not _hf_token:
    _hf_token = os.environ.get('HF_TOKEN', '').strip() or None

if _hf_token:
    try:
        login(_hf_token, add_to_git_credential=False)
        print(f'HF login OK — {whoami()["name"]}')
    except Exception as e:
        print(f'HF login failed: {e}')
        _hf_token = None
else:
    print('No HF_TOKEN — GPQA requires a token (https://huggingface.co/settings/tokens)')


HF login OK — Ani-Sekar


In [ ]:
_GPQA_PREAMBLE = (
    'You are solving a graduate-level science problem. '
    'Before answering: (1) identify relevant physical laws; '
    '(2) define all symbols and state assumptions; '
    '(3) set up the mathematical framework and work through each step; '
    '(4) check dimensional consistency; (5) state the final answer with units.\n\n'
)

if not _hf_token:
    raise RuntimeError(
        'HF_TOKEN required. Set it as a Colab secret or environment variable, '
        'then re-run the login cell.'
    )

print('Loading GPQA Diamond ...')
_ds = load_dataset('Idavidrein/gpqa', 'gpqa_diamond', split='train', token=_hf_token)
_ds = _ds.select(range(min(N_PROBLEMS, len(_ds))))
gpqa_probs = [_GPQA_PREAMBLE + ex['Question'] for ex in _ds]
gpqa_ans   = [ex['Correct Answer'] for ex in _ds]
print(f'  Loaded {len(gpqa_probs)} problems')


Loading GPQA Diamond ...


README.md: 0.00B [00:00, ?B/s]

gpqa_diamond.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/198 [00:00<?, ? examples/s]

  Loaded 30 problems


## §3  Core Dataclasses

In [ ]:
@dataclass
class RKSCConfig:
    tau_similarity_threshold: float = 0.75
    theta_entropy_threshold:  float = 12.0
    cgee_min_exit_layer:      int   = 4
    cgee_stability_eps:       float = 1.0
    cgee_gen_conf_threshold:  float = 0.70
    cgee_gen_conf_gap:        float = 0.06
    cgee_use_relative_gap:    bool  = True
    cgee_relative_gap_thresh: float = 0.08
    cgee_early_exit_decode:   bool  = True
    cgee_min_decode_steps:    int   = 3
    cgee_decode_conf_thresh:  float = 0.72
    cgee_decode_rel_gap:      float = 0.10

@dataclass
class RSBCMConfig:
    max_blocks: int = 2000
    cache_capacity_fraction: float = 0.80

@dataclass
class RKSCSolverConfig:
    rksc:  RKSCConfig  = field(default_factory=RKSCConfig)
    rsbcm: RSBCMConfig = field(default_factory=RSBCMConfig)
    branching_factor:        int  = B_FACTOR
    max_depth:               int  = MAX_DEPTH
    max_new_tokens_per_step: int  = MAX_NEW_TOK
    verify_with_cgee:        bool = True
    verify_full_depth:       bool = False
    max_seq_len:             int  = MAX_SEQ_LEN
    batch_without_kv:        bool = False

def infer_eval_seq_len(model, tokenizer, arch, override=None):
    if override and override > 0:
        return override
    ctx = getattr(model.config, 'max_position_embeddings',
           getattr(model.config, 'n_positions', 2048))
    return min(ctx, MAX_SEQ_LEN)

print('Config dataclasses loaded.')


Config dataclasses loaded.


## §4  Model Loading

In [ ]:
def load_model_and_tokenizer(model_id: str, attn_impl: str = 'sdpa'):
    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = 'left'
    dtype = torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32

    if attn_impl == 'flash_attention_2':
        import importlib
        if importlib.util.find_spec('flash_attn') is None:
            print('FlashAttention2 not found -> falling back to SDPA')
            attn_impl = 'sdpa'

    t0 = time.perf_counter()
    if DEVICE.type == 'cuda':
        m = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype         = dtype,
            device_map          = 'cuda:0',
            attn_implementation = attn_impl,
        )
    else:
        m = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=dtype,
            low_cpu_mem_usage=True,
            attn_implementation=attn_impl)
        m = m.to(DEVICE)
    m.eval()

    if hasattr(m, 'generation_config'):
        m.generation_config.temperature = None
        m.generation_config.top_p       = None
        m.generation_config.top_k       = None
        m.generation_config.do_sample   = False

    elapsed = time.perf_counter() - t0
    if DEVICE.type == 'cuda':
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        used_gb = m.get_memory_footprint() / 1e9
        print(f'  Weights : {used_gb:.1f} GB  GPU free after load: {free_gb:.1f} GB  ({elapsed:.0f}s)')
    print(f'  Loaded  : {model_id} ({dtype}  attn={attn_impl})')
    return m, tok


def diagnose_architecture(model, tokenizer) -> Dict:
    dev = model.get_input_embeddings().weight.device
    inp = tokenizer('hello world', return_tensors='pt').to(dev)
    with torch.no_grad():
        out = model(**inp, output_hidden_states=True)
    hs_ok = (hasattr(out, 'hidden_states') and out.hidden_states is not None
             and len(out.hidden_states) > 1)
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        layer_path, n_layers = 'model.layers', len(model.model.layers)
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
        layer_path, n_layers = 'transformer.h', len(model.transformer.h)
    else:
        layer_path = 'unknown'
        n_layers   = getattr(model.config, 'num_hidden_layers',
                             getattr(model.config, 'n_layer', 12))
    hidden_size = getattr(model.config, 'hidden_size',
                          getattr(model.config, 'n_embd', 768))
    n_heads = getattr(model.config, 'num_attention_heads',
                      getattr(model.config, 'n_head', None))
    n_kv_heads = getattr(model.config, 'num_key_value_heads', n_heads)
    head_dim = getattr(model.config, 'head_dim', None)
    if head_dim is None and hidden_size and n_heads:
        try:
            head_dim = hidden_size // int(n_heads)
        except Exception:
            head_dim = None
    max_pos = getattr(model.config, 'max_position_embeddings',
                      getattr(model.config, 'seq_length', None))
    return dict(hs_in_model_out=hs_ok, n_layers=n_layers,
                hidden_size=hidden_size, vocab_size=model.config.vocab_size,
                layer_path=layer_path,
                num_attention_heads=n_heads,
                num_key_value_heads=n_kv_heads,
                head_dim=head_dim,
                max_position_embeddings=max_pos)


def infer_eval_seq_len(model, tokenizer, arch: Dict, requested: Optional[int] = None) -> int:
    candidates: List[int] = []
    for cand in [
        requested,
        arch.get('max_position_embeddings'),
        getattr(model.config, 'max_position_embeddings', None),
        getattr(model.config, 'seq_length', None),
        getattr(tokenizer, 'model_max_length', None),
    ]:
        if isinstance(cand, (int, np.integer)) and 0 < int(cand) < 100000:
            candidates.append(int(cand))
    if candidates:
        return min(candidates)
    return 2048

print('Model utilities loaded.')


Model utilities loaded.


In [ ]:
for _v in ['model', 'tokenizer', 'ARCH']:
    if _v in globals():
        del globals()[_v]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()
    _free, _total = torch.cuda.mem_get_info()
    print(f'VRAM before load: {(_total-_free)/1e9:.1f} / {_total/1e9:.0f} GB used')
    _needed = MODEL_VRAM_GB.get(MODEL_ID, 16.0)
    if _free/1e9 < _needed:
        raise RuntimeError(f'Need ~{_needed:.0f} GB free, have {_free/1e9:.1f} GB')

print(f'Loading {MODEL_ID} ...')
model, tokenizer = load_model_and_tokenizer(MODEL_ID, attn_impl=ATTN_IMPL)
ARCH = diagnose_architecture(model, tokenizer)
ARCH['_unembed_weight'] = model.get_output_embeddings().weight.data
MAX_SEQ_LEN = infer_eval_seq_len(model, tokenizer, ARCH)

print(f'  Layers={ARCH["n_layers"]}  hidden={ARCH["hidden_size"]}  vocab={ARCH["vocab_size"]}')
if torch.cuda.is_available():
    print(f'  VRAM free after load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')


VRAM before load: 0.4 / 42 GB used
Loading Qwen/Qwen2.5-7B-Instruct ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

  Weights : 15.2 GB  GPU free after load: 26.7 GB  (54s)
  Loaded  : Qwen/Qwen2.5-7B-Instruct (torch.bfloat16  attn=sdpa)
  Layers=28  hidden=3584  vocab=152064
  VRAM free after load: 26.6 GB


## §5  RKSC Solver

```
solve(problem)
├── _root_pass()        → prefix KV cache (1 forward, batch=1)
├── _branch_passes()    → B branches decoded together (batched)
│   ├── _expand_kv(B)   → repeat_interleave KV to batch B
│   └── _stepwise_decode() → token-by-token decode loop
└── _verify()           → batched YES/NO scoring forward pass
    └── _verify_smart() → CGEE gate: skip verify when confident
```


In [ ]:
@dataclass
class BranchSimilarityRecord:
    branch_id:  int
    similarity: float
    kv_reused:  bool


@dataclass
class EarlyExitResult:
    exit_layer:    Optional[int]
    entropy_curve: List[float]
    exited_early:  bool


@dataclass
class BranchResult:
    branch_id:             int
    depth:                 int
    text:                  str
    score:                 Optional[float]
    kv_reused:             bool
    similarity:            float
    cgee_exit_layer:       Optional[int]
    latency_ms:            float
    generation_confidence: float = 0.5
    decode_exit_step:      int   = -1
    problem:               str   = ''


@dataclass
class SolveResult:
    problem:             str
    answer:              str
    branches:            List[BranchResult]
    root_latency_ms:     float
    total_latency_ms:    float
    nowaste_latency_ms:  float

    @property
    def speedup(self) -> float:
        if self.nowaste_latency_ms <= 0:
            return 1.0
        return self.nowaste_latency_ms / self.total_latency_ms

    @property
    def kv_reuse_rate(self) -> float:
        if not self.branches:
            return 0.0
        return sum(b.kv_reused for b in self.branches) / len(self.branches)

    @property
    def mean_cgee_exit_layer(self) -> Optional[float]:
        ls = [b.cgee_exit_layer for b in self.branches if b.cgee_exit_layer is not None]
        return float(np.mean(ls)) if ls else None


print('Data structures loaded.')


Data structures loaded.


In [ ]:
def _squeeze_hs(h):
    h = h.float().nan_to_num(0.0)
    if h.dim() == 3:
        h = h[:, -1, :]
    if h.dim() == 2:
        h = h[0]
    return h


class ASKSManager:
    def __init__(self, cfg: RKSCConfig, arch: Dict):
        self.cfg  = cfg
        self.arch = arch
        self._root_hs_stacked: Optional[torch.Tensor] = None
        self._root_kv = None
        self.records: Dict[int, BranchSimilarityRecord] = {}

    def capture_root(self, kv_cache=None, hidden_states=None):
        if hidden_states:
            vecs = [_squeeze_hs(h).detach() for h in hidden_states]
            self._root_hs_stacked = torch.stack(vecs)
        else:
            self._root_hs_stacked = None
        self._root_kv = kv_cache

    def compute_similarity(self, branch_hs=None):
        if self._root_hs_stacked is None or not branch_hs:
            return 0.0
        dev = self._root_hs_stacked.device
        branch_stacked = torch.stack([_squeeze_hs(h).to(dev) for h in branch_hs])
        cos = F.cosine_similarity(self._root_hs_stacked, branch_stacked, dim=-1)
        return cos.mean().item()

    def score_branch(self, branch_id: int,
                     branch_attn: Optional[List] = None,
                     branch_hs:   Optional[List[torch.Tensor]] = None) -> BranchSimilarityRecord:
        sim   = self.compute_similarity(branch_hs)
        reuse = sim >= self.cfg.tau_similarity_threshold
        rec   = BranchSimilarityRecord(branch_id, sim, reuse)
        self.records[branch_id] = rec
        return rec

    def score_branches_batched(self, branch_attns, branch_hss, branch_ids):
        return [self.score_branch(bid, ba, bh)
                for bid, ba, bh in zip(branch_ids, branch_attns, branch_hss)]

    def get_root_kv(self, branch_id: int):
        rec = self.records.get(branch_id)
        if rec is not None and rec.kv_reused and self._root_kv is not None:
            return self._root_kv
        return None

    def reset(self):
        self._root_hs_stacked = None
        self._root_kv         = None
        self.records          = {}


print('ASKS loaded.')


ASKS loaded.


In [ ]:
class RSBCMManager:
    @dataclass
    class Block:
        block_id:   int
        tree_depth: int
        branch_id:  int
        importance: float

        @property
        def eviction_priority(self) -> float:
            return self.importance / (self.tree_depth + 1)

    def __init__(self, cfg: RSBCMConfig):
        self.cfg   = cfg
        self._pool: Dict[int, RSBCMManager.Block] = {}
        self._next_id        = 0
        self.eviction_events = 0

    def allocate(self, tree_depth: int, branch_id: int,
                 importance: float = 1.0) -> int:
        bid = self._next_id
        self._next_id += 1
        self._pool[bid] = RSBCMManager.Block(bid, tree_depth, branch_id, importance)
        self._maybe_evict()
        return bid

    def _maybe_evict(self):
        if len(self._pool) <= self.cfg.max_blocks:
            return
        n_evict = len(self._pool) - self.cfg.max_blocks
        ordered = sorted(self._pool.values(), key=lambda b: b.eviction_priority)
        for blk in ordered[:n_evict]:
            del self._pool[blk.block_id]
            self.eviction_events += 1

    def evict_branch(self, branch_id: int) -> int:
        victims = [b for b in self._pool.values() if b.branch_id == branch_id]
        for v in victims:
            del self._pool[v.block_id]
        return len(victims)

    @property
    def n_blocks(self) -> int:
        return len(self._pool)

    def reset(self):
        self._pool        = {}
        self._next_id     = 0
        self.eviction_events = 0


print('RSBCM loaded.')


RSBCM loaded.


In [ ]:
class CGEEAnalyzer:
    def __init__(self, cfg: RKSCConfig, arch: Dict):
        self.cfg  = cfg
        self.arch = arch

    def analyze(self, model_output) -> EarlyExitResult:
        if not (hasattr(model_output, 'hidden_states')
                and model_output.hidden_states is not None):
            return EarlyExitResult(None, [], False)

        unembed = self.arch.get('_unembed_weight')
        if unembed is None:
            return EarlyExitResult(None, [], False)

        all_hs   = [h for h in model_output.hidden_states if h is not None]
        layer_hs = all_hs[1:] if len(all_hs) > 1 else all_hs

        raw:       List[torch.Tensor] = []
        valid_idx: List[int]          = []
        for i, hs in enumerate(layer_hs):
            h = hs[:, -1, :].float().nan_to_num(0.0)
            if h.norm() > 0:
                raw.append(h.squeeze(0))
                valid_idx.append(i)

        if not raw:
            return EarlyExitResult(None, [], False)

        dev       = unembed.device
        all_h     = torch.stack(raw).to(dev)
        unembed_f = unembed.float()
        logits    = F.linear(all_h, unembed_f)
        logits    = logits.clamp(-1000.0, 1000.0)
        probs     = logits.softmax(dim=-1)
        entropies = -(probs * (probs + 1e-10).log()).sum(dim=-1)
        ent_list  = entropies.tolist()

        entropy_curve: List[float] = []
        exit_layer:    Optional[int] = None

        for layer_idx, ent in zip(valid_idx, ent_list):
            if not np.isfinite(ent):
                continue
            entropy_curve.append(ent)
            n = len(entropy_curve)
            if (exit_layer is None
                    and layer_idx >= self.cfg.cgee_min_exit_layer
                    and ent < self.cfg.theta_entropy_threshold
                    and n >= 2
                    and abs(ent - entropy_curve[-2]) < self.cfg.cgee_stability_eps):
                exit_layer = layer_idx

        return EarlyExitResult(
            exit_layer    = exit_layer,
            entropy_curve = entropy_curve,
            exited_early  = exit_layer is not None,
        )


print('CGEE loaded.')


CGEE loaded.


In [ ]:
_BRANCH_HINTS = [
    "Use first principles and dimensional analysis.",
    "Apply conservation laws and symmetry arguments.",
    "Work backwards from the answer choices.",
    "Set up equations and solve algebraically.",
    "Use a limiting case or special case to verify.",
    "Draw a diagram or use geometric reasoning.",
    "Use order-of-magnitude estimation first.",
    "Apply the most relevant formula directly.",
]

_CGEE_EXIT_HISTORY: list = []

def _ms(t0: float) -> float:
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000


class _EarlyExitSignal(Exception):
    def __init__(self, layer_idx: int):
        self.layer_idx = layer_idx


class RKSCSolverV8:
    """
    RKSC solver: shared prefix KV cache + step-wise batched decoding.

    Core design:
    - Prefix KV computed once; expanded to B branches via repeat_interleave.
    - Branches process only suffix tokens, sharing all GPU decode kernels.
    - CGEE gates the verify pass on generation confidence.
    """

    def __init__(self, model, tokenizer, cfg: RKSCSolverConfig,
                 arch: Dict, disable_reuse: bool = False):
        self.model         = model
        self.tok           = tokenizer
        self.cfg           = cfg
        self.arch          = arch
        self.disable_reuse = disable_reuse
        self._dev          = model.get_input_embeddings().weight.device
        self._max_seq_len  = infer_eval_seq_len(
            model, tokenizer, arch, getattr(cfg, 'max_seq_len', None)
        )

        if '_unembed_weight' not in arch:
            arch['_unembed_weight'] = model.get_output_embeddings().weight.data

        self.asks  = ASKSManager(cfg.rksc, arch)
        self.rsbcm = RSBCMManager(cfg.rsbcm)
        self.cgee  = CGEEAnalyzer(cfg.rksc, arch)
        self._root_layer_importances: List[float] = []
        self._root_pkv_obj   = None
        self._prefix_len     = 0
        self._n_solved       = 0

        self._cgee_handles: list = []

        _yn = self.tok
        self._yes_id: Optional[int] = next(
            (ids[0] for ids in [_yn.encode(' YES', add_special_tokens=False),
                                _yn.encode('YES',  add_special_tokens=False)] if ids),
            None)
        self._no_id:  Optional[int] = next(
            (ids[0] for ids in [_yn.encode(' NO',  add_special_tokens=False),
                                _yn.encode('NO',   add_special_tokens=False)] if ids),
            None)

    def _get_transformer_layers(self, layer_indices=None):
        lp = self.arch.get('layer_path', '')
        if lp == 'model.layers' and hasattr(self.model, 'model'):
            all_layers = list(self.model.model.layers)
        elif lp == 'transformer.h' and hasattr(self.model, 'transformer'):
            all_layers = list(self.model.transformer.h)
        else:
            all_layers = [m for n, m in self.model.named_modules()
                          if n.count('.') == 2 and any(k in n for k in ('layers', '.h'))]
        if layer_indices is not None:
            return [all_layers[i] for i in layer_indices if i < len(all_layers)]
        return all_layers


    def remove_hooks(self):
        """Remove all CGEE hooks. Call before discarding a solver."""
        for h in self._cgee_handles:
            h.remove()
        self._cgee_handles.clear()

    def _tok(self, text: str, **kwargs):
        return self.tok(text, return_tensors='pt',
                        truncation=True, max_length=self._max_seq_len,
                        **kwargs).to(self._dev)

    def _shared_prefix(self, problem: str) -> str:
        return f'Solve step by step:\n{problem}\nLet us reason step by step.'

    def _branch_suffix(self, branch_idx: int) -> str:
        hint = _BRANCH_HINTS[branch_idx % len(_BRANCH_HINTS)]
        return f'\nApproach #{branch_idx + 1}: {hint}\nStep 1:'

    def _expand_kv(self, pkv, B: int):
        """
        Expand a batch-1 KV cache to batch B via repeat_interleave.
        Uses .clone() on each tensor — faster than deepcopy, avoids
        expand() which can trigger contiguity issues in attention kernels.
        """
        if pkv is None:
            return None
        from transformers import DynamicCache
        try:
            new_cache = DynamicCache()
            if hasattr(pkv, 'key_cache') and len(pkv.key_cache) > 0:
                for li in range(len(pkv.key_cache)):
                    new_cache.update(
                        pkv.key_cache[li].repeat_interleave(B, dim=0),
                        pkv.value_cache[li].repeat_interleave(B, dim=0),
                        layer_idx=li)
            elif hasattr(pkv, 'layers') and len(pkv.layers) > 0:
                for li, layer in enumerate(pkv.layers):
                    new_cache.update(
                        layer.keys.repeat_interleave(B, dim=0),
                        layer.values.repeat_interleave(B, dim=0),
                        layer_idx=li)
            else:
                import copy
                return copy.deepcopy(pkv)
            return new_cache
        except Exception as _e:
            print(f'[V8] _expand_kv: {_e}', flush=True)
            import copy
            return copy.deepcopy(pkv)

    def _kv_seq_len(self, pkv) -> int:
        """Return number of tokens currently in the KV cache."""
        if pkv is None:
            return 0
        if hasattr(pkv, 'key_cache') and pkv.key_cache:
            return pkv.key_cache[0].shape[2]
        if isinstance(pkv, tuple) and pkv and pkv[0]:
            return pkv[0][0].shape[2]
        return 0

    def _stepwise_decode(self,
                         input_ids:       torch.Tensor,
                         attention_mask:  torch.Tensor,
                         past_key_values,
                         max_new_tokens:  int,
                         early_exit_cfg:  Optional[tuple] = None,
                         ) -> Tuple[torch.Tensor, float]:
        """Decode using a pre-filled KV cache via a direct model() loop."""
        B      = input_ids.shape[0]
        dev    = self._dev
        eos_id = self.tok.eos_token_id or 0

        total_ctx   = attention_mask.shape[1]
        max_buf     = total_ctx + max_new_tokens
        full_mask   = torch.ones(B, max_buf, dtype=torch.long, device=dev)
        _mask_is_dense = bool(attention_mask.all())
        if not _mask_is_dense:
            full_mask[:, :total_ctx] = attention_mask  # only copy if padding present
        generated   = torch.full((B, max_new_tokens), eos_id,
                                  dtype=torch.long, device=dev)
        finished    = torch.zeros(B, dtype=torch.bool, device=dev)

        t0         = time.perf_counter()
        pkv        = past_key_values
        curr       = total_ctx
        _conf_list: list = []  # top-1 softmax probability per step per branch

        _ee_enabled   = (early_exit_cfg is not None and B > 1)
        _ee_min_steps = early_exit_cfg[0] if _ee_enabled else max_new_tokens
        _ee_conf      = early_exit_cfg[1] if _ee_enabled else 1.1
        _ee_rel_gap   = early_exit_cfg[2] if _ee_enabled else 1.1
        _exit_step    = -1  # -1 = no early exit

        with torch.inference_mode():
            out      = self.model(
                input_ids       = input_ids,
                attention_mask  = full_mask[:, :curr],
                past_key_values = pkv,
                use_cache       = True,
            )
            next_tok = out.logits[:, -1:, :].argmax(dim=-1)   # [B, 1]
            _conf_list.append(
                out.logits[:, -1, :].float().softmax(-1).max(-1).values.cpu())
            pkv      = out.past_key_values
            generated[:, 0] = next_tok[:, 0]
            finished |= (next_tok[:, 0] == eos_id)
            curr     += 1

            for step in range(1, max_new_tokens):
                if finished.all():
                    break
                out      = self.model(
                    input_ids       = next_tok,                  # [B, 1]
                    attention_mask  = full_mask[:, :curr],       # view, no alloc
                    past_key_values = pkv,
                    use_cache       = True,
                )
                next_tok = out.logits[:, -1:, :].argmax(dim=-1)
                _conf_list.append(
                    out.logits[:, -1, :].float().softmax(-1).max(-1).values.cpu())
                pkv      = out.past_key_values
                generated[:, step] = next_tok[:, 0]
                finished |= (next_tok[:, 0] == eos_id)
                curr     += 1

                if _ee_enabled and step >= _ee_min_steps and step % 2 == 0:
                    _stacked   = torch.stack(_conf_list)         # [steps, B]
                    _br_means  = _stacked.mean(0)               # [B] rolling mean
                    _sorted, _ = _br_means.sort(descending=True)
                    _top, _sec = float(_sorted[0]), float(_sorted[1])
                    _rel_lead  = (_top - _sec) / max(_top, 1e-6)
                    if _top >= _ee_conf and _rel_lead >= _ee_rel_gap:
                        finished[:] = True  # stop all branches
                        _exit_step  = step
                        break

        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        _mean_conf = (torch.stack(_conf_list).mean(0)
                      if _conf_list else torch.full((B,), 0.5))
        return generated, (time.perf_counter() - t0) * 1000, _mean_conf, _exit_step


    def _root_pass(self, problem: str):
        """Compute prefix KV cache. Probes both paths and caches the faster decision."""
        prefix_text = self._shared_prefix(problem)
        prefix_inp  = self._tok(prefix_text)
        prefix_len  = prefix_inp['input_ids'].shape[1]
        self._prefix_len = prefix_len

        if self.disable_reuse:
            self._root_pkv_obj = None
            return "", 0.0, prefix_len

        if not hasattr(self, '_kv_probe_cache'):
            self._kv_probe_cache: Dict[int, bool] = {}
        _bucket = prefix_len // 64   # re-probe if prefix changes by >64 tokens

        if _bucket not in self._kv_probe_cache:
            _sfx_text = self._branch_suffix(0)
            _sfx_inp  = self.tok(_sfx_text, return_tensors='pt',
                                  truncation=True, max_length=128).to(self._dev)
            _full_inp = self._tok(prefix_text + _sfx_text)

            _ts_full = []
            for _ in range(3):
                t0 = time.perf_counter()
                with torch.inference_mode():
                    self.model(**_full_inp, use_cache=True,
                               output_hidden_states=False, output_attentions=False)
                _ts_full.append(_ms(t0))
            _ms_full = min(_ts_full)

            with torch.inference_mode():
                _fwd = self.model(**prefix_inp, use_cache=True,
                                  output_hidden_states=False, output_attentions=False)
            _pkv_probe = _fwd.past_key_values; del _fwd
            _ts_pre = []
            for _ in range(3):
                t0 = time.perf_counter()
                with torch.inference_mode():
                    _fwd2 = self.model(**prefix_inp, use_cache=True,
                                       output_hidden_states=False, output_attentions=False)
                _ts_pre.append(_ms(t0))
                del _fwd2.past_key_values, _fwd2
            _ms_pre = min(_ts_pre)

            _attn_sfx = torch.cat([
                torch.ones(1, prefix_len, dtype=torch.long, device=self._dev),
                torch.ones(1, _sfx_inp['input_ids'].shape[1],
                           dtype=torch.long, device=self._dev),
            ], dim=1)
            _ts_sfx = []
            for _ in range(3):
                t0 = time.perf_counter()
                with torch.inference_mode():
                    self.model(_sfx_inp['input_ids'], attention_mask=_attn_sfx,
                               past_key_values=_pkv_probe, use_cache=True)
                _ts_sfx.append(_ms(t0))
            _ms_sfx = min(_ts_sfx)

            B = self.cfg.branching_factor


            _sfx_ids_B  = _sfx_inp['input_ids'].expand(B, -1)
            _pfx_ids_B  = prefix_inp['input_ids'].expand(B, -1)
            _full_ids_B = torch.cat([_pfx_ids_B, _sfx_ids_B], dim=1)
            _full_msk_B = torch.ones(B, _full_ids_B.shape[1],
                                     dtype=torch.long, device=self._dev)
            _ts_batch_full = []
            for _ in range(3):
                t0 = time.perf_counter()
                with torch.inference_mode():
                    self.model(input_ids=_full_ids_B,
                               attention_mask=_full_msk_B,
                               use_cache=False,
                               output_hidden_states=False,
                               output_attentions=False)
                _ts_batch_full.append(_ms(t0))
            _ms_batch_full = min(_ts_batch_full)

            _attn_sfx_B = torch.cat([
                torch.ones(B, prefix_len, dtype=torch.long, device=self._dev),
                torch.ones(B, _sfx_inp['input_ids'].shape[1],
                           dtype=torch.long, device=self._dev),
            ], dim=1)
            _ts_batch_sfx = []
            for _ in range(3):
                _pkv_B = self._expand_kv(_pkv_probe, B)
                t0 = time.perf_counter()
                with torch.inference_mode():
                    self.model(_sfx_ids_B,
                               attention_mask=_attn_sfx_B,
                               past_key_values=_pkv_B,
                               use_cache=True)
                _ts_batch_sfx.append(_ms(t0))
                del _pkv_B
            _ms_batch_sfx = min(_ts_batch_sfx)

            _kv_net     = _ms_batch_full - (_ms_pre + _ms_batch_sfx)
            _beneficial = _kv_net > 0

            _per_branch_saving = _ms_full - _ms_sfx
            if _per_branch_saving > 0:
                _breakeven_B    = _ms_pre / _per_branch_saving
                _breakeven_str  = f'breakeven_B≈{_breakeven_B:.1f}'
            else:
                _breakeven_B    = float('inf')
                _breakeven_str  = 'breakeven_B=∞ (KV load > recompute on SDPA)'

            self._kv_probe_cache[_bucket] = _beneficial
            if not hasattr(self, '_kv_probe_measurements'):
                self._kv_probe_measurements: dict = {}
            self._kv_probe_measurements[_bucket] = dict(
                prefix_len=prefix_len,
                ms_full_B1=_ms_full, ms_prefix_fwd=_ms_pre,
                ms_sfx_B1=_ms_sfx,
                ms_batch_full=_ms_batch_full, ms_batch_sfx=_ms_batch_sfx,
                per_branch_saving_ms=_per_branch_saving,
                breakeven_B=_breakeven_B, beneficial=_beneficial,
            )
            print(f'  KV probe [prefix≈{prefix_len}tok B={B} bucket={_bucket}]: '
                  f'batch_full={_ms_batch_full:.1f}ms  '
                  f'prefix_fwd={_ms_pre:.1f}ms  '
                  f'batch_sfx_w_kv={_ms_batch_sfx:.1f}ms  '
                  f'net={_kv_net:+.1f}ms  '
                  f'{_breakeven_str}  '
                  f'→ {"BENEFICIAL" if _beneficial else "SKIP (batched-no-KV faster)"}')
            del _pkv_probe
            if self._dev.type == 'cuda': torch.cuda.empty_cache()

        _kv_beneficial = self._kv_probe_cache[_bucket]

        if not _kv_beneficial:
            self._root_pkv_obj = None
            return "", 0.0, prefix_len

        t0 = time.perf_counter()
        with torch.inference_mode():
            fwd = self.model(
                **prefix_inp,
                output_hidden_states = False,
                output_attentions    = False,
                use_cache            = True,
            )
        ms = _ms(t0)
        self._root_pkv_obj = fwd.past_key_values
        del fwd
        return "", ms, prefix_len

    def _branch_passes(self, problem: str, root_text: str,
                       shared_prefix_len: int, depth: int) -> List[BranchResult]:
        """Generate all B branches — batched with KV sharing (RKSC) or serial (baseline)."""
        B           = self.cfg.branching_factor
        prefix_text = self._shared_prefix(problem)
        dev         = self._dev
        pad_id      = self.tok.pad_token_id or self.tok.eos_token_id

        suffix_ids_list = []
        for b in range(B):
            sfx = self.tok(
                self._branch_suffix(b),
                return_tensors='pt',
                truncation=True,
                max_length=128,
            )['input_ids']
            suffix_ids_list.append(sfx)

        max_sfx = max(s.shape[1] for s in suffix_ids_list)
        padded_sfx = torch.full((B, max_sfx), pad_id, dtype=torch.long, device=dev)
        sfx_attn   = torch.zeros(B, max_sfx, dtype=torch.long, device=dev)
        for i, ids in enumerate(suffix_ids_list):
            L = ids.shape[1]
            padded_sfx[i, max_sfx - L:] = ids[0].to(dev)   # right-align
            sfx_attn[i, max_sfx - L:]   = 1

        kv_reused = False
        batch_kv  = None

        _bucket_cur  = shared_prefix_len // 64
        _probe_cache = getattr(self, '_kv_probe_cache', {})
        _kv_ok_here  = _probe_cache.get(_bucket_cur, True)  # default True if not probed
        _force_batch_no_kv = (
            not self.disable_reuse
            and not _kv_ok_here
        )

        if not self.disable_reuse and not _force_batch_no_kv \
                and self._root_pkv_obj is not None:
            try:
                batch_kv  = self._expand_kv(self._root_pkv_obj, B)
                kv_reused = (batch_kv is not None)
            except Exception as _e:
                print(f'[V8] expand_kv: {_e}', flush=True)
                batch_kv  = None
                kv_reused = False

        if kv_reused:
            prefix_ones = torch.ones(B, shared_prefix_len, dtype=torch.long, device=dev)
            first_pass_mask = torch.cat([prefix_ones, sfx_attn], dim=1)
            input_for_decode = padded_sfx  # [B, max_sfx]

            t0 = time.perf_counter()
            _ee_cfg = (
                (self.cfg.rksc.cgee_min_decode_steps,
                 self.cfg.rksc.cgee_decode_conf_thresh,
                 self.cfg.rksc.cgee_decode_rel_gap)
                if self.cfg.verify_with_cgee and
                   getattr(self.cfg.rksc, 'cgee_early_exit_decode', True)
                else None
            )
            gen_ids, gen_ms, _branch_conf, _exit_step = self._stepwise_decode(
                input_ids       = input_for_decode,
                attention_mask  = first_pass_mask,
                past_key_values = batch_kv,
                max_new_tokens  = self.cfg.max_new_tokens_per_step,
                early_exit_cfg  = _ee_cfg,
            )
        else:
            prefix_ids = self.tok(
                prefix_text, return_tensors='pt',
                truncation=True, max_length=self._max_seq_len,
            )['input_ids'].to(dev)

            if getattr(self.cfg, 'batch_without_kv', False) or _force_batch_no_kv:
                max_full = max(
                    prefix_ids.shape[1] + s.shape[1] for s in suffix_ids_list)
                batch_ids  = torch.full((B, max_full), pad_id,
                                        dtype=torch.long, device=dev)
                batch_mask = torch.zeros(B, max_full, dtype=torch.long, device=dev)
                for b, sfx_b in enumerate(suffix_ids_list):
                    full_b = torch.cat([prefix_ids, sfx_b.to(dev)], dim=1)
                    fl = full_b.shape[1]
                    batch_ids[b, :fl]  = full_b[0]
                    batch_mask[b, :fl] = 1
                _ee_cfg2 = (
                    (self.cfg.rksc.cgee_min_decode_steps,
                     self.cfg.rksc.cgee_decode_conf_thresh,
                     self.cfg.rksc.cgee_decode_rel_gap)
                    if self.cfg.verify_with_cgee and
                       getattr(self.cfg.rksc, 'cgee_early_exit_decode', True)
                    else None
                )
                gen_ids, gen_ms, _branch_conf, _exit_step = self._stepwise_decode(
                    input_ids       = batch_ids,
                    attention_mask  = batch_mask,
                    past_key_values = None,
                    max_new_tokens  = self.cfg.max_new_tokens_per_step,
                    early_exit_cfg  = _ee_cfg2,
                )
            else:
                gen_ids_list  = []
                gen_ms        = 0.0
                _serial_confs = []
                for b in range(B):
                    sfx_b  = suffix_ids_list[b].to(dev)
                    full_b = torch.cat([prefix_ids, sfx_b], dim=1)
                    attn_b = torch.ones(1, full_b.shape[1],
                                        dtype=torch.long, device=dev)
                    t_b = time.perf_counter()
                    g_ids, _, _b_conf = self._stepwise_decode(
                        input_ids       = full_b,
                        attention_mask  = attn_b,
                        past_key_values = None,
                        max_new_tokens  = self.cfg.max_new_tokens_per_step,
                    )
                    gen_ms += _ms(t_b)
                    gen_ids_list.append(g_ids[0])
                    _serial_confs.append(_b_conf[0])
                gen_ids      = torch.stack(gen_ids_list)
                _branch_conf = torch.stack(_serial_confs)  # [B]
                _exit_step   = -1  # serial path: no early exit

        texts = []
        for b in range(B):
            t = self.tok.decode(gen_ids[b], skip_special_tokens=True).strip()
            texts.append(t)

        try:
            _bconf = [float(_branch_conf[b]) for b in range(B)]
        except Exception:
            _bconf = [0.5] * B

        branches = []
        for b in range(B):
            sim = 1.0 if kv_reused else 0.0
            self.asks.records[b] = BranchSimilarityRecord(b, sim, kv_reused)
            importance = 1.0
            if self._root_layer_importances:
                li = min(depth - 1, len(self._root_layer_importances) - 1)
                importance = self._root_layer_importances[li]
            self.rsbcm.allocate(depth, b, importance=importance)
            branches.append(BranchResult(b, depth, texts[b], None,
                                         kv_reused, sim, None, gen_ms / B,
                                         generation_confidence=_bconf[b],
                                         decode_exit_step=_exit_step,
                                         problem=problem))
        return branches

    def _verify_cgee_batch(self, branches: List[BranchResult],
                           problem: str = '') -> List[BranchResult]:
        """
        Batched CGEE verify: one padded forward pass for all B branches.
        At exit layer, scores by YES/NO binary probability (not layer heuristic).
        Single PCIe transfer for all entropy and logit values.
        """
        global _CGEE_EXIT_HISTORY

        _prob_ctx = problem[:2000] if problem else ''
        prompts  = [
            f'Problem: {_prob_ctx}\n\n'
            f'Proposed answer: {b.text[:400]}\n\n'
            f'Is this answer correct? YES or NO.'
            for b in branches
        ]
        batch_enc = self.tok(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=self._max_seq_len,
        ).to(self._dev)
        padded   = batch_enc['input_ids']
        attn_msk = batch_enc['attention_mask'].long()
        lens     = attn_msk.sum(dim=-1).tolist()

        t0 = time.perf_counter()
        with torch.inference_mode():
            out = self.model(input_ids=padded, attention_mask=attn_msk,
                             output_hidden_states=True, use_cache=False)
        ms_batch = _ms(t0)

        theta     = self.cfg.rksc.theta_entropy_threshold
        eps       = self.cfg.rksc.cgee_stability_eps
        min_l     = self.cfg.rksc.cgee_min_exit_layer
        n_layers  = self.arch['n_layers']
        unembed_f = self.arch['_unembed_weight'].float()
        layer_hs  = out.hidden_states[1:] if out.hidden_states is not None else []
        del out

        if layer_hs:
            _lens_t  = torch.tensor(lens, device=self._dev) - 1
            _lasts   = torch.stack(
                [hs[torch.arange(len(branches)), _lens_t, :] for hs in layer_hs]
            ).float().nan_to_num(0.0)          # [n_L, B, hidden]
            _logits  = F.linear(_lasts, unembed_f).clamp(-1e3, 1e3)
            _probs   = _logits.softmax(dim=-1)  # [n_L, B, vocab]
            _ent_all = -(_probs * (_probs + 1e-10).log()).sum(dim=-1)  # [n_L, B]
            _ent_np  = _ent_all.cpu().float().numpy()  # single PCIe transfer
            del _lasts, _logits, _probs, _ent_all
        else:
            _ent_np = None

        _yes_id, _no_id = self._yes_id, self._no_id
        _lens_t = torch.tensor(lens, device=self._dev) - 1  # last real token per branch
        if layer_hs and _yes_id is not None and _no_id is not None:
            _lasts_all = torch.stack(
                [hs[torch.arange(len(branches)), _lens_t, :] for hs in layer_hs]
            ).float().nan_to_num(0.0)
            _logits_all = F.linear(_lasts_all, unembed_f).clamp(-1e3, 1e3)  # [n_L, B, vocab]
            _yn_logits  = _logits_all[:, :, [_yes_id, _no_id]]  # [n_L, B, 2]
            _yn_np      = _yn_logits.cpu().float().numpy()       # single PCIe transfer
            del _lasts_all, _logits_all, _yn_logits
        else:
            _yn_np = None

        for i, branch in enumerate(branches):
            exit_layer: Optional[int]   = None
            prev_ent:   Optional[float] = None
            if _ent_np is not None:
                for l_idx in range(len(layer_hs)):
                    ent = float(_ent_np[l_idx, i])
                    if not np.isfinite(ent): prev_ent = None; continue
                    if (exit_layer is None and l_idx >= min_l
                            and ent < theta and prev_ent is not None
                            and abs(ent - prev_ent) < eps):
                        exit_layer = l_idx; break
                    prev_ent = ent
            branch.cgee_exit_layer = exit_layer
            branch.latency_ms     += ms_batch / len(branches)

            _score_layer = exit_layer if exit_layer is not None else len(layer_hs) - 1
            if _yn_np is not None and _score_layer < _yn_np.shape[0]:
                logit_y = float(_yn_np[_score_layer, i, 0])
                logit_n = float(_yn_np[_score_layer, i, 1])
                branch.score = float(
                    torch.tensor([logit_y, logit_n]).softmax(0)[0].item())
            else:
                branch.score = 0.5  # no information

            if exit_layer is not None:
                _CGEE_EXIT_HISTORY.append(exit_layer)

        return branches

    def _verify_smart(self, branches: List[BranchResult],
                      problem: str = '') -> List[BranchResult]:
        """
        Verify-pass CGEE: skip the verify forward pass when generation is decisive.
        Complements decode-level CGEE (which saves decode steps).
        Two conditions (either triggers skip):
          A. top_conf >= thresh AND abs_gap >= gap
          B. top_conf >= 0.9×thresh AND rel_gap >= rel_thresh
        """
        if not hasattr(self, '_cgee_skip_count'):
            self._cgee_skip_count = 0; self._cgee_call_count = 0
        self._cgee_call_count += 1

        if len(branches) < 2:
            branches[0].score = branches[0].generation_confidence
            self._cgee_skip_count += 1
            return branches

        confs    = [b.generation_confidence for b in branches]
        sc       = sorted(confs, reverse=True)
        top, sec = sc[0], sc[1]
        abs_gap  = top - sec
        rel_gap  = abs_gap / max(top, 1e-6)

        thresh    = self.cfg.rksc.cgee_gen_conf_threshold
        gap       = self.cfg.rksc.cgee_gen_conf_gap
        use_rel   = getattr(self.cfg.rksc, 'cgee_use_relative_gap', True)
        rel_t     = getattr(self.cfg.rksc, 'cgee_relative_gap_thresh', 0.08)

        skip = (top >= thresh and abs_gap >= gap)
        if use_rel and not skip:
            skip = (top >= thresh * 0.90 and rel_gap >= rel_t)

        if skip:
            for b in branches: b.score = b.generation_confidence
            self._cgee_skip_count += 1
            return branches
        return self._verify_batch_full(branches, problem=problem)

    def _verify_batch_full(self, branches: List[BranchResult],
                           problem: str = '') -> List[BranchResult]:
        """
        Batched full-depth verify scored by YES/NO next-token probability.
        Includes problem context so the model evaluates correctness, not fluency.
        """
        _prob_ctx = problem[:2000] if problem else ''
        prompts  = [
            f'Problem: {_prob_ctx}\n\nProposed answer: {b.text[:400]}\n\n'
            f'Is this answer correct? YES or NO.'
            for b in branches
        ]
        batch_enc = self.tok(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=self._max_seq_len,
        ).to(self._dev)
        padded   = batch_enc['input_ids']
        attn_msk = batch_enc['attention_mask'].long()
        lens     = attn_msk.sum(dim=-1).tolist()

        t0 = time.perf_counter()
        with torch.inference_mode():
            out = self.model(input_ids=padded, attention_mask=attn_msk,
                             output_hidden_states=False, use_cache=False)
        ms_batch = _ms(t0)

        _yes_id, _no_id = self._yes_id, self._no_id

        _tok_idx   = torch.tensor([l-1 for l in lens], device=self._dev)
        _batch_idx = torch.arange(len(branches), device=self._dev)
        last_logits = out.logits[_batch_idx, _tok_idx, :].float()
        del out

        for i, branch in enumerate(branches):
            if _yes_id is not None and _no_id is not None:
                logit_yes = last_logits[i, _yes_id].item()
                logit_no  = last_logits[i, _no_id].item()
                branch.score = float(torch.tensor([logit_yes, logit_no]).softmax(0)[0].item())
            else:
                p = last_logits[i].softmax(dim=-1)
                branch.score = float(p.max().item())
            branch.latency_ms += ms_batch / len(branches)
        del last_logits
        return branches

    def solve(self, problem: str) -> SolveResult:
        self.asks.reset()
        self.rsbcm.reset()
        t_total = time.perf_counter()
        self._n_solved += 1

        root_text, root_ms, shared_prefix_len = self._root_pass(problem)

        branches = self._branch_passes(problem, root_text, shared_prefix_len, depth=1)

        if self.cfg.verify_with_cgee:
            branches = self._verify_smart(branches, problem=problem)
        else:
            branches = self._verify_batch_full(branches, problem=problem)

        best = max(branches, key=lambda b: b.score or 0.0)

        if not hasattr(self, '_decode_exit_history'):
            self._decode_exit_history: list = []
        _es = branches[0].decode_exit_step if branches else -1
        self._decode_exit_history.append(_es)

        self._root_pkv_obj = None

        return SolveResult(
            problem            = problem,
            answer             = best.text,
            branches           = branches,
            root_latency_ms    = root_ms,
            total_latency_ms   = _ms(t_total),
            nowaste_latency_ms = 0.0,
        )


RKSCSolver = RKSCSolverV8
print('RKSCSolver ready.')


RKSCSolver ready.


## §6  Table 1 — Latency Decomposition

Three conditions, all measured vs a **Batched-no-KV** reference (B branches batched, no prefix KV sharing):

| Condition | What it isolates |
|---|---|
| **KV only** (RKSC ASKS, no verify skip) | Pure KV-sharing gain |
| **CGEE only** (no KV, verify-skip active) | Pure verify-skip gain |
| **RKSC** (KV + CGEE combined) | End-to-end throughput gain |

Problems are padded to a **1024-token shared prefix** — at this length, prefix attention (O(n²)) dominates and KV sharing saves one full prefill per problem across all B branches.


In [ ]:
PREFIX_TARGET = 1024
_FILLER = ('Shared context: state all assumptions explicitly and '
              'preserve dimensional units throughout each step. ')

# Measure filler cost once so _pad_to_prefix can bulk-prepend instead of looping.
_FILLER_TOKS = len(tokenizer.encode(_FILLER, add_special_tokens=False))

def _pad_to_prefix(problem: str, target: int = PREFIX_TARGET) -> str:
    """Pad problem with neutral filler to reach target shared-prefix token count.
    Uses a closed-form estimate: measures filler tokens once, bulk-prepends, then
    fine-tunes ≤ a few iterations — no 600-step tokenization loop.
    """
    _tpl = 'Solve step by step:\n{}\nLet us reason step by step.'
    aug      = problem.strip()
    hard_cap = MAX_SEQ_LEN - 256
    target_c = min(target, hard_cap)

    current = tokenizer(_tpl.format(aug), return_tensors='pt',
                        truncation=True, max_length=MAX_SEQ_LEN
                        )['input_ids'].shape[1]
    if current >= target_c:
        return aug

    n_reps  = max(1, int(np.ceil((target_c - current) / max(_FILLER_TOKS, 1))))
    aug     = _FILLER * n_reps + aug

    for _ in range(20):   # fine-tune: at most a handful of iterations
        current = tokenizer(_tpl.format(aug), return_tensors='pt',
                            truncation=True, max_length=MAX_SEQ_LEN
                            )['input_ids'].shape[1]
        if current >= target_c:
            break
        aug = _FILLER + aug
    return aug

print(f'Padding {len(gpqa_probs)} problems to ~{PREFIX_TARGET}-token shared prefix ...')
probs_padded   = [_pad_to_prefix(p) for p in gpqa_probs]
_t1_lens  = [
    tokenizer(f'Solve step by step:\n{p}\nLet us reason step by step.',
              return_tensors='pt', truncation=True,
              max_length=MAX_SEQ_LEN)['input_ids'].shape[1]
    for p in probs_padded
]
print(f'Shared prefix — mean: {np.mean(_t1_lens):.0f} tok  '
      f'min: {min(_t1_lens)}  max: {max(_t1_lens)}')
print(f'At prefix≈{np.mean(_t1_lens):.0f} tokens with B={B_FACTOR}, '
      f'KV probe should find caching beneficial.')


Padding 30 problems to ~1024-token shared prefix ...
Shared prefix — mean: 1032 tok  min: 1024  max: 1038
At prefix≈1032 tokens with B=8, KV probe should find caching beneficial.


In [ ]:
def benchmark_condition(model, tokenizer, problems, cfg, arch,
                         disable_reuse, n_runs=2, label='',
                         full_depth_verify=False, batch_no_kv=False):
    cfg2 = deepcopy(cfg)
    cfg2.batch_without_kv = batch_no_kv
    if disable_reuse:
        cfg2.rksc.tau_similarity_threshold = 1.1
        cfg2.verify_full_depth = full_depth_verify
        cfg2.verify_with_cgee  = False
    else:
        cfg2.verify_full_depth = full_depth_verify

    solver = RKSCSolverV8(model, tokenizer, cfg2, arch, disable_reuse=disable_reuse)
    lats, reuse_rates = [], []

    for prob in tqdm(problems, desc=label or f'disable={disable_reuse}'):
        run_lats = []
        last     = None
        for _ in range(n_runs):
            res = solver.solve(prob)
            run_lats.append(res.total_latency_ms)
            last = res
            del res
        lats.append(float(np.mean(run_lats)))
        if last:
            reuse_rates.append(last.kv_reuse_rate)
        del last
        if len(lats) % 5 == 0:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    solver.remove_hooks()
    del solver
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return lats, reuse_rates


In [ ]:
cal    = CALIBRATED.get(MODEL_ID, {'tau': 0.75, 'theta': 12.0})
cfg_bm = RKSCSolverConfig(
    branching_factor        = B_FACTOR,
    max_depth               = MAX_DEPTH,
    max_new_tokens_per_step = MAX_NEW_TOK,
)
cfg_bm.rksc.tau_similarity_threshold = cal['tau']
cfg_bm.rksc.theta_entropy_threshold  = cal['theta']

def _gpu_free():
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()
        return torch.cuda.mem_get_info()[0] / 1e9
    return 0.0

def _run_condition(label, ckpt_key, cfg_override=None,
                   disable=False, full_depth=False, batch_no_kv=False):
    _vkey = f'{ckpt_key}_{CKPT_VERSION}'.replace('/', '_').replace('|', '_')
    cached = ckpt_load(_vkey)
    if cached is not None:
        print(f'{label}: loaded from cache', flush=True)
        return cached['lats'], cached['rr']
    cfg_use = deepcopy(cfg_override or cfg_bm)
    print(f'{label}: running ... (VRAM {_gpu_free():.1f} GB free)', flush=True)
    lats, rr = benchmark_condition(
        model, tokenizer, probs_padded, cfg_use, ARCH,
        disable_reuse=disable, n_runs=N_RUNS,
        full_depth_verify=full_depth, batch_no_kv=batch_no_kv, label=label)
    ckpt_save(_vkey, {'lats': lats, 'rr': rr})
    print(f'  Done. mean={np.mean(lats):.1f} ms')
    return lats, rr

lat_batch_nokv, _ = _run_condition(
    'Batched-no-KV (reference)', 'lat_batch_nokv',
    disable=True, full_depth=True, batch_no_kv=True)

cfg_kv = deepcopy(cfg_bm)
cfg_kv.verify_with_cgee  = False
cfg_kv.verify_full_depth = True
lat_kv, rr_kv = _run_condition('KV only', 'lat_kv', cfg_kv)

cfg_cgee = deepcopy(cfg_bm)
cfg_cgee.verify_with_cgee  = True
cfg_cgee.verify_full_depth = False
lat_cgee, rr_cgee = _run_condition(
    'CGEE only', 'lat_cgee', cfg_cgee, disable=True, batch_no_kv=True)

lat_rksc, rr_rksc = _run_condition('RKSC (KV + CGEE)', 'lat_rksc')


Batched-no-KV (reference): running ... (VRAM 26.6 GB free)


Batched-no-KV (reference): 100%|██████████| 30/30 [01:56<00:00,  3.88s/it]


  Done. mean=1912.4 ms
KV only: running ... (VRAM 26.6 GB free)


KV only:   0%|          | 0/30 [00:00<?, ?it/s]

  KV probe [prefix≈1033tok B=8 bucket=16]: batch_full=583.3ms  prefix_fwd=88.8ms  batch_sfx_w_kv=40.0ms  net=+454.6ms  breakeven_B≈1.6  → BENEFICIAL


...

  Done. mean=1908.7 ms
RKSC (KV + CGEE): running ... (VRAM 26.6 GB free)


RKSC (KV + CGEE):   0%|          | 0/30 [00:00<?, ?it/s]

  KV probe [prefix≈1033tok B=8 bucket=16]: batch_full=581.6ms  prefix_fwd=88.8ms  batch_sfx_w_kv=40.0ms  net=+452.8ms  breakeven_B≈1.7  → BENEFICIAL


RKSC (KV + CGEE): 100%|██████████| 30/30 [01:24<00:00,  2.80s/it]

  Done. mean=1377.3 ms


In [ ]:
def speedup(sys_lats, base_lats):
    """Ratio-of-means speedup and percent saved (algebraically consistent)."""
    sp  = float(np.mean(base_lats)) / float(np.mean(sys_lats))
    pct = (1.0 - 1.0 / sp) * 100.0
    return sp, pct

_, p_kv   = stats.ttest_rel(lat_kv,   lat_batch_nokv)
_, p_cgee = stats.ttest_rel(lat_cgee, lat_batch_nokv)
_, p_rksc = stats.ttest_rel(lat_rksc, lat_batch_nokv)

ref_ms = np.mean(lat_batch_nokv)

rows = []
for label, lats, rr, p in [
    ('Batched-no-KV (reference)', lat_batch_nokv, [0.0]*len(lat_batch_nokv), None),
    ('KV only',                   lat_kv,         rr_kv,                    p_kv),
    ('CGEE only',                 lat_cgee,        rr_cgee,                  p_cgee),
    ('RKSC (KV + CGEE)',          lat_rksc,        rr_rksc,                  p_rksc),
]:
    sp, pct = speedup(lats, lat_batch_nokv)
    rows.append({
        'Condition':    label,
        'Mean (ms)':    f'{np.mean(lats):.1f}±{np.std(lats):.1f}',
        'Speedup':      f'{sp:.3f}×' if sp > 1.001 else '1.000× (ref)',
        '% saved':      f'{pct:.1f}%' if sp > 1.001 else '—',
        'KV reuse':     f'{np.mean(rr):.0%}' if any(r > 0 for r in rr) else '—',
        'p-value':      f'{p:.4f}' if p is not None else '—',
    })

df_t1 = pd.DataFrame(rows)
print()
print('=' * 75)
print('TABLE 1 — Latency Decomposition')
print(f'Model: {MODEL_ID}   B={B_FACTOR}  tok={MAX_NEW_TOK}  N={N_PROBLEMS}  prefix≈1024')
print(f'Reference: Batched-no-KV  |  all speedups measured against this baseline')
print('=' * 75)
print(df_t1.to_string(index=False))
print()

sp_kv,   pct_kv   = speedup(lat_kv,   lat_batch_nokv)
sp_cgee, pct_cgee = speedup(lat_cgee, lat_batch_nokv)
sp_rksc, pct_rksc = speedup(lat_rksc, lat_batch_nokv)

print('Speedup decomposition:')
print(f'  KV sharing alone  : {sp_kv:.3f}×  ({ref_ms - np.mean(lat_kv):.0f} ms saved, {pct_kv:.1f}%)  p={p_kv:.4f}')
print(f'  CGEE alone        : {sp_cgee:.3f}×  ({ref_ms - np.mean(lat_cgee):.0f} ms saved, {pct_cgee:.1f}%)  p={p_cgee:.4f}')
print(f'  RKSC combined     : {sp_rksc:.3f}×  ({ref_ms - np.mean(lat_rksc):.0f} ms saved, {pct_rksc:.1f}%)  p={p_rksc:.4f}')

_gap = (ref_ms - np.mean(lat_rksc)) - ((ref_ms - np.mean(lat_kv)) + (ref_ms - np.mean(lat_cgee)))
_add_label = 'sub' if _gap < 0 else 'super'
print(f'  Additivity gap    : {_gap:+.1f} ms  ({_add_label}-additive)')



TABLE 1 — Latency Decomposition
Model: Qwen/Qwen2.5-7B-Instruct   B=8  tok=32  N=30  prefix≈1024
Reference: Batched-no-KV  |  all speedups measured against this baseline
                Condition    Mean (ms)      Speedup % saved KV reuse p-value
Batched-no-KV (reference)  1912.4±29.7 1.000× (ref)       —        —       —
                  KV only 1460.1±239.1       1.310×   23.7%     100%  0.0000
                CGEE only   1908.7±7.3       1.002×    0.2%        —  0.5553
         RKSC (KV + CGEE) 1377.3±353.6       1.389×   28.0%     100%  0.0000

Speedup decomposition:
  KV sharing alone  : 1.310×  (452 ms saved, 23.7%)  p=0.0000
  CGEE alone        : 1.002×  (4 ms saved, 0.2%)  p=0.5553
  RKSC combined     : 1.389×  (535 ms saved, 28.0%)  p=0.0000
  Additivity gap    : +79.1 ms  (super-additive)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

labels  = ['Batched-no-KV\n(ref)', 'KV only', 'CGEE only', 'RKSC\n(KV+CGEE)']
all_lats = [lat_batch_nokv, lat_kv, lat_cgee, lat_rksc]
means    = [np.mean(l) for l in all_lats]
stds     = [np.std(l)  for l in all_lats]
colors   = ['#aec6e8', '#1f77b4', '#ff7f0e', '#2ca02c']

ax = axes[0]
bars = ax.bar(labels, means, yerr=stds, capsize=5,
              color=colors, edgecolor='white', linewidth=0.8)
ref = np.mean(lat_batch_nokv)
for bar, m, name in zip(bars, means, labels):
    tag = '1.000×' if 'ref' in name else f'{ref/m:.3f}×'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stds)*0.1,
            tag, ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Wall-clock latency (ms)', fontsize=12)
ax.set_title(f'Table 1 — Latency per condition\n{MODEL_ID.split("/")[-1]}  B={B_FACTOR}  tok={MAX_NEW_TOK}', fontsize=10)
ax.set_ylim(0, max(means) * 1.30)
ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
save_kv   = ref - np.mean(lat_kv)
save_cgee = ref - np.mean(lat_cgee)
save_rksc = ref - np.mean(lat_rksc)
bars2 = ax2.bar(['KV only', 'CGEE only', 'RKSC\n(KV+CGEE)'],
                [save_kv, save_cgee, save_rksc],
                color=['#1f77b4', '#ff7f0e', '#2ca02c'],
                edgecolor='white', linewidth=0.8, width=0.5)
for bar, val in zip(bars2, [save_kv, save_cgee, save_rksc]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val:.0f} ms\n({val/ref*100:.1f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Latency saved vs Batched-no-KV (ms)', fontsize=12)
ax2.set_title('Speedup attribution\n(all vs Batched-no-KV baseline)', fontsize=10)
ax2.set_ylim(0, max(save_kv, save_cgee, save_rksc) * 1.6)
ax2.grid(axis='y', alpha=0.3)
ax2.axhline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()
plt.close('all')


## §7  Extended Evaluation — Multi-Model, Multi-Dataset

Tests across 7B–14B GQA models and four datasets spanning difficulty tiers:

| Dataset | Difficulty | Expected CGEE skip rate |
|---|---|---|
| GSM8K | Easy (math) | 60–80% |
| ARC-Challenge | Medium | 40–60% |
| MMLU-STEM | Medium-hard | 20–40% |
| GPQA Diamond | Hard (grad-level) | 10–30% |


In [ ]:
EVAL_MODELS_PRIORITY = [
    'Qwen/Qwen2.5-7B-Instruct',
    'mistralai/Mistral-7B-Instruct-v0.3',
    'tiiuae/Falcon3-7B-Instruct',
    'tiiuae/Falcon3-10B-Instruct',
    'meta-llama/Llama-3-8B-Instruct',
]

if torch.cuda.is_available():
    _vram_free = torch.cuda.mem_get_info()[0] / 1e9
    EVAL_MODELS = [m for m in EVAL_MODELS_PRIORITY
                   if MODEL_VRAM_GB.get(m, 30.0) + 2.0 <= _vram_free]
    print(f'VRAM available : {_vram_free:.1f} GB')
    print(f'Models selected: {len(EVAL_MODELS)}')
    for m in EVAL_MODELS:
        print(f'  {m}  ({MODEL_VRAM_GB.get(m, 0):.0f} GB)')
else:
    EVAL_MODELS = []
    print('No CUDA — extended eval requires GPU')

_GPQA_PREAMBLE_EXT = (
    'You are solving a graduate-level science problem. '
    'Reason carefully and show all steps.\n\n'
)

EVAL_DATASETS = {
    'gpqa': {
        'hf_name': 'Idavidrein/gpqa', 'config': 'gpqa_diamond', 'split': 'train',
        'q_field': 'Question', 'a_field': 'Correct Answer',
        'preamble': _GPQA_PREAMBLE_EXT,
    },
    'gsm8k': {
        'hf_name': 'openai/gsm8k', 'config': 'main', 'split': 'test',
        'q_field': 'question', 'a_field': 'answer',
        'preamble': 'Solve the following math problem step by step.\n\n',
    },
    'arc_challenge': {
        'hf_name': 'allenai/ai2_arc', 'config': 'ARC-Challenge', 'split': 'test',
        'q_field': 'question', 'a_field': 'answerKey',
        'preamble': 'Answer this multiple-choice science question. Think step by step.\n\n',
    },
    'mmlu_stem': {
        'hf_name': 'cais/mmlu', 'config': 'college_physics', 'split': 'test',
        'q_field': 'question', 'a_field': 'answer',
        'preamble': 'Answer this university-level physics question.\n\n',
    },
}

N_EVAL      = 50
N_EVAL_RUNS = 1
print(f'\nN_EVAL={N_EVAL}  tok={MAX_NEW_TOK_EXT}  B={B_FACTOR}')
_est = len(EVAL_MODELS) * N_EVAL * 4 * 12 / 60
print(f'Est. runtime: ~{_est:.0f} min  (4 conditions × {len(EVAL_MODELS)} models)')


VRAM available : 26.6 GB
Models selected: 5
  Qwen/Qwen2.5-7B-Instruct  (15 GB)
  mistralai/Mistral-7B-Instruct-v0.3  (15 GB)
  tiiuae/Falcon3-7B-Instruct  (15 GB)
  tiiuae/Falcon3-10B-Instruct  (20 GB)
  meta-llama/Llama-3-8B-Instruct  (15 GB)

N_EVAL=50  tok=8  B=8
Est. runtime: ~200 min  (4 conditions × 5 models)


In [ ]:
def load_eval_dataset(ds_name: str, ds_cfg: Dict, n: int):
    """Returns (questions: List[str], ground_truth: List[str])."""
    preamble = ds_cfg.get('preamble', '')
    try:
        kw = dict(split=ds_cfg['split'])
        if ds_cfg.get('config'):
            kw['name'] = ds_cfg['config']
        ds = load_dataset(ds_cfg['hf_name'], **kw)
        ds = ds.select(range(min(n, len(ds))))
        probs = [preamble + str(ex[ds_cfg['q_field']]) for ex in ds]
        a_field = ds_cfg.get('a_field')
        ans   = [str(ex[a_field]) if a_field and a_field in ex else '' for ex in ds]
        print(f'  {ds_name}: loaded {len(probs)} problems')
        return probs, ans
    except Exception as e:
        raise RuntimeError(
            f'Dataset "{ds_name}" failed to load: {type(e).__name__}: {e}'
        ) from e

print('Loading extended eval datasets ...')
EVAL_DATA:       Dict[str, List[str]] = {}
EVAL_GT_ANSWERS: Dict[str, List[str]] = {}   # ground-truth answers, keyed by dataset name
for ds_name, ds_cfg in EVAL_DATASETS.items():
    if ds_name == 'gpqa' and 'gpqa_probs' in globals() and gpqa_probs:
        EVAL_DATA[ds_name] = gpqa_probs[:N_EVAL]
        EVAL_GT_ANSWERS[ds_name] = (gpqa_ans[:N_EVAL]
                                     if 'gpqa_ans' in globals() else [])
        print(f'  gpqa: reusing {len(EVAL_DATA[ds_name])} pre-loaded problems')
    else:
        _qs, _as = load_eval_dataset(ds_name, ds_cfg, N_EVAL)
        EVAL_DATA[ds_name]       = _qs
        EVAL_GT_ANSWERS[ds_name] = _as
    avg_tok = np.mean(tokenizer(EVAL_DATA[ds_name], truncation=True,
                               max_length=MAX_SEQ_LEN,
                               return_length=True)['length'])
    print(f'  {ds_name}: {len(EVAL_DATA[ds_name])} problems, avg_tokens={avg_tok:.0f}')


Loading extended eval datasets ...
  gpqa: reusing 30 pre-loaded problems
  gpqa: 30 problems, avg_tokens=182


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

...

college_physics/dev-00000-of-00001.parqu(…):   0%|          | 0.00/4.51k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/102 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

  mmlu_stem: loaded 50 problems
  mmlu_stem: 50 problems, avg_tokens=50


In [ ]:
def _pad_for_extended_eval(problem: str, tokenizer, target: int = 1024,
                            max_seq_len: int = 2048) -> str:
    """Pad problem text to ~1024 tokens, matching Table 1 methodology."""
    _FILLER = (
        "Shared context note: keep assumptions explicit, preserve units, "
        "and treat the setup carefully. "
    )
    _tpl = "Solve step by step:\n{}\nLet us reason step by step."

    def _n_tok(text):
        return int(tokenizer(text, return_tensors="pt",
                             truncation=True, max_length=max_seq_len
                             )["input_ids"].shape[1])

    hard_cap = max_seq_len - 256
    target_c = min(target, hard_cap)
    aug = problem.strip()
    current = _n_tok(_tpl.format(aug))
    if current >= target_c:
        return aug
    filler_toks = max(_n_tok(_FILLER), 1)
    n_reps = max(1, int(np.ceil((target_c - current) / filler_toks)))
    aug = _FILLER * n_reps + aug
    for _ in range(20):
        if _n_tok(_tpl.format(aug)) >= target_c:
            break
        aug = _FILLER + aug
    return aug


def run_extended_benchmark(model_id: str, datasets: Dict[str, List[str]],
                            n_runs: int = 1,
                            gt_answers: Optional[Dict[str, List[str]]] = None) -> Dict:
    """
    Benchmark each model across all datasets under four conditions:
    Batched-no-KV (reference), RKSC (KV only), RKSC+verify, RKSC+CGEE.
    Returns a dict of per-dataset result dicts.
    """
    print(f'\n{"="*60}\nModel: {model_id}\n{"="*60}')

    for _v in ['_ext_model', '_ext_tok']:
        if _v in globals(): del globals()[_v]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()

    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    needed  = MODEL_VRAM_GB.get(model_id, 30.0)
    if free_gb < needed + 2.0:
        print(f'  SKIPPED: {free_gb:.1f} GB free, need {needed:.0f}+2 GB')
        return {}

    _ext_model, _ext_tok = load_model_and_tokenizer(model_id, attn_impl=ATTN_IMPL)
    _ext_arch = diagnose_architecture(_ext_model, _ext_tok)
    _ext_arch['_unembed_weight'] = _ext_model.get_output_embeddings().weight.data
    _cal = CALIBRATED.get(model_id, {'tau': 0.82, 'theta': 6.0})

    _n_layers = _ext_arch.get('n_layers', '?')
    _n_kv     = _ext_arch.get('num_key_value_heads', '?')
    _n_q      = _ext_arch.get('num_attention_heads', '?')
    _hd       = _ext_arch.get('head_dim', '?')
    print(f'  Arch: {_n_layers} layers  {_n_q}Q/{_n_kv}KV heads  head_dim={_hd}')

    try:
        _kv_mb_per_branch = (2 * int(_n_layers) * int(_n_kv) * int(_hd)
                             * 1024 * 2 / 1e6)
        print(f'  KV cache per branch @ prefix=1024: {_kv_mb_per_branch:.0f} MB  ')
        print(f'  KV sharing saves {(B_FACTOR-1)*_kv_mb_per_branch:.0f} MB vs B={B_FACTOR} separate caches')
    except Exception:
        pass

    results = {}
    for ds_name, problems in datasets.items():
        print(f'\n  Dataset: {ds_name} ({len(problems)} problems)')

        problems_padded = [
            _pad_for_extended_eval(p, _ext_tok, target=1024,
                                    max_seq_len=MAX_SEQ_LEN)
            for p in problems
        ]
        avg_raw = float(np.mean(_ext_tok(
            problems, truncation=True, max_length=MAX_SEQ_LEN,
            return_length=True)['length']))
        avg_pad = float(np.mean(_ext_tok(
            problems_padded, truncation=True, max_length=MAX_SEQ_LEN,
            return_length=True)['length']))
        print(f'    prefix tokens: raw={avg_raw:.0f}  padded={avg_pad:.0f}  (target=1024)')

        _cfg = RKSCSolverConfig(
            branching_factor=B_FACTOR, max_depth=MAX_DEPTH,
            max_new_tokens_per_step=MAX_NEW_TOK_EXT,
            max_seq_len=MAX_SEQ_LEN)
        _cfg.rksc.tau_similarity_threshold = _cal['tau']
        _cfg.rksc.theta_entropy_threshold  = _cal['theta']

        try:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            lat_batch_nkv, _ = benchmark_condition(
                _ext_model, _ext_tok, problems_padded, _cfg, _ext_arch,
                disable_reuse=True, batch_no_kv=True,
                full_depth_verify=False,
                n_runs=n_runs, label=f'  {ds_name} batch-no-KV')
            _peak_batch_mb = (torch.cuda.max_memory_allocated() / 1e6
                              if torch.cuda.is_available() else 0.0)

            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            _cfg_asks = deepcopy(_cfg)
            _cfg_asks.verify_with_cgee  = False
            _cfg_asks.verify_full_depth = False
            lat_rksc, rr = benchmark_condition(
                _ext_model, _ext_tok, problems_padded, _cfg_asks, _ext_arch,
                disable_reuse=False, full_depth_verify=False,
                n_runs=n_runs, label=f'  {ds_name} RKSC')
            _peak_rksc_mb = (torch.cuda.max_memory_allocated() / 1e6
                              if torch.cuda.is_available() else 0.0)
            _mem_saved_mb = _peak_batch_mb - _peak_rksc_mb

            _cfg_verify = deepcopy(_cfg)
            _cfg_verify.verify_with_cgee  = False
            _cfg_verify.verify_full_depth = True
            lat_rksc_verify, _ = benchmark_condition(
                _ext_model, _ext_tok, problems_padded, _cfg_verify, _ext_arch,
                disable_reuse=False, full_depth_verify=True,
                n_runs=n_runs, label=f'  {ds_name} RKSC+verify')

            _cfg_cgee = deepcopy(_cfg)
            _cfg_cgee.verify_with_cgee  = True
            _cfg_cgee.verify_full_depth = False
            _solver_cgee = RKSCSolverV8(
                _ext_model, _ext_tok, _cfg_cgee, _ext_arch, disable_reuse=False)
            _cgee_lats = []
            for _prob in tqdm(problems_padded, desc=f'  {ds_name} RKSC+CGEE', leave=False):
                _res = _solver_cgee.solve(_prob)
                _cgee_lats.append(_res.total_latency_ms)
            _cgee_skip_n  = getattr(_solver_cgee, '_cgee_skip_count', 0)
            _cgee_call_n  = getattr(_solver_cgee, '_cgee_call_count', 1)
            _cgee_skip_rate = _cgee_skip_n / max(_cgee_call_n, 1)
            _cgee_per_problem = getattr(_solver_cgee, '_cgee_skip_log', [])
            _solver_cgee.remove_hooks()
            del _solver_cgee
            lat_rksc_cgee = _cgee_lats

            _acc_cgee_answers    = []
            _acc_verify_answers  = []
            _acc_gt              = (gt_answers or {}).get(ds_name, [])
            _solver_ans_cgee   = RKSCSolverV8(
                _ext_model, _ext_tok, _cfg_cgee, _ext_arch, disable_reuse=False)
            _solver_ans_verify = RKSCSolverV8(
                _ext_model, _ext_tok, _cfg_verify, _ext_arch, disable_reuse=False)
            for _pi, _prob in enumerate(
                    tqdm(problems_padded, desc=f'  {ds_name} accuracy-check', leave=False)):
                _r_cgee   = _solver_ans_cgee.solve(_prob)
                _r_verify = _solver_ans_verify.solve(_prob)
                _acc_cgee_answers.append(getattr(_r_cgee,   'answer', '') or '')
                _acc_verify_answers.append(getattr(_r_verify, 'answer', '') or '')
            _solver_ans_cgee.remove_hooks();   del _solver_ans_cgee
            _solver_ans_verify.remove_hooks(); del _solver_ans_verify

            _agree = [a.strip() == b.strip()
                      for a, b in zip(_acc_cgee_answers, _acc_verify_answers)]
            _agreement_rate = sum(_agree) / max(len(_agree), 1)

            def _extract_gsm8k_answer(text: str) -> str:
                """Extract the number after '####' in GSM8K ground-truth strings."""
                import re as _re
                m = _re.search(r'####\s*([\d,\.\-]+)', text)
                return m.group(1).replace(',', '').strip() if m else text.strip()

            def _extract_mmlu_answer(gt_raw, choices) -> str:
                """Convert MMLU integer answer index (0-3) to 'A. <choice text>'."""
                _letters = ['A', 'B', 'C', 'D']
                try:
                    idx = int(gt_raw)
                    letter = _letters[idx]
                    choice_text = choices[idx] if choices and idx < len(choices) else ''
                    return f'{letter}. {choice_text}'.strip()
                except (ValueError, IndexError, TypeError):
                    return str(gt_raw).strip()

            def _acc_vs_gt(answers, gt_list, ds_name_='', choices_list=None):
                """Format-aware accuracy check.

                * GSM8K  — extracts number after '####'; compares last number in
                  model output vs extracted GT number (handles comma-formatting).
                * ARC / MMLU — regex word-boundary match on letter (A/B/C/D) so
                  'the answer is B' matches 'B' without false positives from words
                  like 'Be', 'By', etc.
                * Default fallback — original substring match.
                """
                import re as _re
                if not gt_list:
                    return float('nan'), []
                hits = []
                for i, (ans, gt) in enumerate(zip(answers, gt_list)):
                    ans_s = ans.strip()
                    gt_s  = str(gt).strip()
                    if ds_name_ == 'gsm8k':
                        gt_num = _extract_gsm8k_answer(gt_s)
                        nums = _re.findall(r'[\-]?\d+(?:[,\.]\d+)*', ans_s)
                        pred_num = nums[-1].replace(',', '') if nums else ''
                        try:
                            hit = (float(pred_num) == float(gt_num))
                        except ValueError:
                            hit = (pred_num == gt_num)
                    elif ds_name_ in ('arc_challenge', 'mmlu_stem'):
                        choices_ = (choices_list[i]
                                    if choices_list and i < len(choices_list)
                                    else None)
                        gt_norm = (_extract_mmlu_answer(gt_s, choices_)
                                   if ds_name_ == 'mmlu_stem'
                                   else gt_s.upper().strip())
                        letter = gt_norm[0] if gt_norm else ''
                        hit = bool(letter and _re.search(
                            rf'\b{_re.escape(letter)}\b', ans_s, _re.IGNORECASE))
                    else:
                        gt_l   = gt_s.lower()
                        ans_l  = ans_s.lower()
                        hit    = bool(gt_l and (gt_l in ans_l or ans_l in gt_l))
                    hits.append(hit)
                return sum(hits) / max(len(hits), 1), hits

            _cgee_acc,   _cgee_hits   = _acc_vs_gt(_acc_cgee_answers,   _acc_gt, ds_name)
            _verify_acc, _verify_hits = _acc_vs_gt(_acc_verify_answers, _acc_gt, ds_name)
            _acc_delta = (_cgee_acc - _verify_acc) if (_cgee_acc == _cgee_acc
                                                        and _verify_acc == _verify_acc) else float('nan')

            _disagree_idxs = [i for i, ok in enumerate(_agree) if not ok]
            _cgee_caused_errors = (
                [i for i in _disagree_idxs
                 if _acc_gt and not _cgee_hits[i] and _verify_hits[i]]
                if _acc_gt else []
            )

            print(f'    CGEE accuracy report:')
            print(f'      Attempts made       : {_cgee_call_n} calls across {len(problems_padded)} problems')
            print(f'      Skipped             : {_cgee_skip_n}/{_cgee_call_n} '
                  f'({_cgee_skip_rate:.1%}) verify passes skipped')
            print(f'      CGEE vs full-verify agreement: {_agreement_rate:.1%} '
                  f'({len(_agree)-len(_disagree_idxs)}/{len(_agree)} problems)')
            if _disagree_idxs:
                print(f'      Disagreements       : {len(_disagree_idxs)} problems where CGEE changed answer')
                for _di in _disagree_idxs[:3]:
                    print(f'        [prob {_di}] CGEE: {_acc_cgee_answers[_di][:60]!r}')
                    print(f'                verify: {_acc_verify_answers[_di][:60]!r}')
            if _acc_gt:
                print(f'      Accuracy vs GT — CGEE: {_cgee_acc:.1%}  '
                      f'full-verify: {_verify_acc:.1%}  '
                      f'delta: {_acc_delta:+.1%}')
                if _cgee_caused_errors:
                    print(f'      ⚠  CGEE caused {len(_cgee_caused_errors)} errors '
                          f'(full-verify was correct but CGEE was not)'
                          f' on problems: {_cgee_caused_errors}')
                else:
                    print(f'      ✓  CGEE introduced 0 errors vs full-verify on ground truth')

            _mean_batch       = float(np.mean(lat_batch_nkv))
            _mean_rksc        = float(np.mean(lat_rksc))
            _mean_rksc_verify = float(np.mean(lat_rksc_verify))
            _mean_rksc_cgee   = float(np.mean(lat_rksc_cgee))

            speedup_kv  = _mean_batch  / _mean_rksc
            savings_ms  = _mean_batch  - _mean_rksc
            savings_pct = (1.0 - 1.0 / speedup_kv) * 100.0

            _verify_cost_ms   = _mean_rksc_verify - _mean_rksc        # cost of running full verify
            _cgee_saved_ms    = _mean_rksc_verify - _mean_rksc_cgee   # ms saved by CGEE skipping
            _cgee_speedup_vs_verify = (_mean_rksc_verify / _mean_rksc_cgee
                                       if _mean_rksc_cgee > 0 else 1.0)

            results[ds_name] = dict(
                lat_batch_nokv=list(lat_batch_nkv),
                lat_rksc=list(lat_rksc),
                lat_rksc_verify=list(lat_rksc_verify),
                lat_rksc_cgee=list(lat_rksc_cgee),
                speedup_kv_mean=speedup_kv,
                speedup_kv_std=float(np.std(
                    np.array(lat_batch_nkv) / np.array(lat_rksc))),
                savings_ms_mean=savings_ms,
                savings_ms_std=float(np.std(
                    np.array(lat_batch_nkv) - np.array(lat_rksc))),
                savings_pct=savings_pct,
                kv_reuse_rate=float(np.mean(rr)),
                verify_cost_ms=_verify_cost_ms,
                cgee_saved_ms=_cgee_saved_ms,
                cgee_skip_rate=_cgee_skip_rate,
                cgee_skip_n=_cgee_skip_n,
                cgee_call_n=_cgee_call_n,
                cgee_speedup_vs_verify=_cgee_speedup_vs_verify,
                cgee_agreement_rate=_agreement_rate,
                cgee_n_disagree=len(_disagree_idxs),
                cgee_n_caused_errors=len(_cgee_caused_errors),
                cgee_accuracy_gt=_cgee_acc,
                verify_accuracy_gt=_verify_acc,
                accuracy_delta=_acc_delta,
                has_ground_truth=bool(_acc_gt),
                n_problems=len(problems_padded),
                avg_tokens_raw=avg_raw,
                avg_tokens_padded=avg_pad,
                model_n_layers=_n_layers,
                model_n_kv_heads=_n_kv,
                model_n_q_heads=_n_q,
                model_head_dim=_hd,
                mem_saved_mb=_mem_saved_mb,
                max_new_tokens_used=MAX_NEW_TOK_EXT,
            )
            r = results[ds_name]

            print(f'    batch-no-KV   : {_mean_batch:7.0f} ms  (reference, no verify)')
            print(f'    RKSC no-verify: {_mean_rksc:7.0f} ms  KV gain={speedup_kv:.3f}x ({savings_pct:.1f}%)')
            print(f'    RKSC+verify   : {_mean_rksc_verify:7.0f} ms  verify costs {_verify_cost_ms:+.0f} ms')
            print(f'    RKSC+CGEE     : {_mean_rksc_cgee:7.0f} ms  skip={_cgee_skip_rate:.0%}  ')
            print(f'      CGEE saved  : {_cgee_saved_ms:.0f} ms vs full-verify ')
            print(f'      ({_cgee_skip_n}/{_cgee_call_n} verify passes skipped)')
            if _cgee_skip_rate < 0.10 and _cgee_call_n >= 10:
                print(f'      ⚠  Low skip rate — consider recalibrating cgee_gen_conf_threshold')
            if _cgee_call_n < 20:
                _ci = 1.96 * (_cgee_skip_rate * (1 - _cgee_skip_rate) / max(_cgee_call_n, 1)) ** 0.5
                print(f'      ⚠  N={_cgee_call_n} calls: CI on skip rate ±{_ci:.0%} — run ≥30 problems')

        except Exception:
            import traceback; traceback.print_exc()
            results[ds_name] = {}
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return results


In [ ]:
print('Freeing primary model ...')

extended_results = ckpt_load('extended_results') or {}

_stale = [mid for mid, r in extended_results.items()
          if isinstance(r, dict) and r and
          any(isinstance(ds, dict) and ds.get('avg_tokens_padded', 0) < 800
              for ds in r.values() if isinstance(ds, dict))]
if _stale:
    print(f'Invalidating stale results (prefix<1024): {_stale}')
    for mid in _stale:
        del extended_results[mid]

for _mid in EVAL_MODELS:
    if _mid in extended_results and extended_results[_mid]:
        r0 = next((v for v in extended_results[_mid].values()
                   if isinstance(v, dict) and 'avg_tokens_padded' in v), {})
        if (r0.get('avg_tokens_padded', 0) >= 800
                and r0.get('max_new_tokens_used', 32) == MAX_NEW_TOK_EXT):
            print(f'  {_mid}: loaded from checkpoint (prefix={r0.get("avg_tokens_padded", 0):.0f} tok)')
            continue

    if torch.cuda.is_available():
        _pre_free = torch.cuda.mem_get_info()[0] / 1e9
        _needed   = MODEL_VRAM_GB.get(_mid, 30.0) + 2.0
        print(f'  [{_mid.split("/")[-1]}] VRAM: {_pre_free:.1f} GB free, '
              f'need {_needed:.0f} GB', flush=True)
        if _pre_free < _needed:
            print(f'    SKIPPING — insufficient VRAM '
                  f'({_pre_free:.1f} < {_needed:.0f} GB required)')
            extended_results[_mid] = {}
            continue

    try:
        extended_results[_mid] = run_extended_benchmark(
            _mid, EVAL_DATA, n_runs=N_EVAL_RUNS,
            gt_answers=EVAL_GT_ANSWERS)
        ckpt_save('extended_results', extended_results)
    except torch.cuda.OutOfMemoryError:
        print(f'  SKIPPED {_mid}: GPU OOM — try reducing B_FACTOR or N_EVAL')
        extended_results[_mid] = {}
    except Exception as _e:
        import traceback; traceback.print_exc()
        extended_results[_mid] = {}

print(f'Reloading primary model: {MODEL_ID}')
model, tokenizer = load_model_and_tokenizer(MODEL_ID, attn_impl=ATTN_IMPL)
ARCH = diagnose_architecture(model, tokenizer)
ARCH['_unembed_weight'] = model.get_output_embeddings().weight.data
print(f'Primary model ready.  VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')


Freeing primary model ...
  [Qwen2.5-7B-Instruct] VRAM: 26.6 GB free, need 17 GB

Model: Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  Weights : 15.2 GB  GPU free after load: 11.4 GB  (6s)
  Loaded  : Qwen/Qwen2.5-7B-Instruct (torch.bfloat16  attn=sdpa)
  Arch: 28 layers  28Q/4KV heads  head_dim=128
  KV cache per branch @ prefix=1024: 59 MB  
  KV sharing saves 411 MB vs B=8 separate caches

  Dataset: gpqa (30 problems)
    prefix tokens: raw=182  padded=1016  (target=1024)


  gpqa RKSC:   0%|          | 0/30 [00:00<?, ?it/s]

  KV probe [prefix≈1034tok B=8 bucket=16]: batch_full=581.4ms  prefix_fwd=88.7ms  batch_sfx_w_kv=40.0ms  net=+452.8ms  breakeven_B≈1.6  → BENEFICIAL


...

  mmlu_stem accuracy-check:   0%|          | 0/50 [00:00<?, ?it/s]

  KV probe [prefix≈1025tok B=8 bucket=16]: batch_full=582.6ms  prefix_fwd=88.6ms  batch_sfx_w_kv=39.4ms  net=+454.6ms  breakeven_B≈1.6  → BENEFICIAL
  KV probe [prefix≈1025tok B=8 bucket=16]: batch_full=581.6ms  prefix_fwd=88.8ms  batch_sfx_w_kv=39.5ms  net=+453.3ms  breakeven_B≈1.6  → BENEFICIAL


    CGEE accuracy report:
      Attempts made       : 50 calls across 50 problems
      Skipped             : 13/50 (26.0%) verify passes skipped
      CGEE vs full-verify agreement: 74.0% (37/50 problems)
      Disagreements       : 13 problems where CGEE changed answer
        [prob 2] CGEE: 'Consider a reversible isothermal expansion of an'
                verify: 'Define the key terms and principles.\n-'
        [prob 4] CGEE: 'Identify the relevant conservation laws and symmet'
                verify: 'Identify the key concepts.\nThe question is'
        [prob 7] CGEE: 'Estimate the kinetic energy at 0.'
                verify: 'Identify the given information and the goal.'
      Accuracy vs GT — CGEE: 0.0%  full-verify: 0.0%  delta: +0.0%
      ✓  CGEE introduced 0 errors vs full-verify on ground truth
    batch-no-KV   :    1113 ms  (reference, no verify)
    RKSC no-verify:     663 ms  KV gain=1.678x (40.4%)
    RKSC+verify   :     662 ms  verify costs -1 ms
    RKSC+CGEE     :

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 55.44 MiB is free. Including non-PyTorch memory, this process has 39.43 GiB memory in use. Of the allocated memory 38.92 GiB is allocated by PyTorch, and 5.57 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print('\n' + '='*100)
print('EXTENDED EVALUATION: Multi-Model, Multi-Dataset')
print(f'Reference: Batched-no-KV  |  tok={MAX_NEW_TOK_EXT}  prefix≈1024 tok  B={B_FACTOR}')
print('Speedup = mean(Batch-noKV) / mean(condition)  [ratio-of-means]')
print('='*100)

_W = 35
_hdr = (f'  {"Model":<{_W}} {"Dataset":<13}'
        f' {"Batch-noKV":>11} {"KV":>9} {"CGEE":>9} {"RKSC":>9}'
        f' {"KV gain":>9} {"RKSC gain":>10}'
        f' {"CGEE skip%":>11} {"Arch":>14}')
print(_hdr)
print('  ' + '-'*130)

rows_ext = []
for mid, ds_results in extended_results.items():
    model_short = mid.split('/')[-1]
    for ds_name, r in ds_results.items():
        if not r or ds_name.startswith('_') or not isinstance(r, dict):
            continue

        lat_b    = float(np.mean(r['lat_batch_nokv'])) if 'lat_batch_nokv' in r else float('nan')
        lat_kv_  = float(np.mean(r['lat_rksc']))       if 'lat_rksc'       in r else float('nan')
        lat_cgee_= float(np.mean(r.get('lat_rksc_cgee', [float('nan')])))
        lat_rksc_= float(np.mean(r.get('lat_rksc_cgee', [float('nan')])))

        _lat_kv_only    = float(np.mean(r['lat_rksc']))          if 'lat_rksc'          in r else float('nan')
        _lat_rksc_verify= float(np.mean(r['lat_rksc_verify']))   if 'lat_rksc_verify'   in r else float('nan')
        _lat_rksc_cgee  = float(np.mean(r['lat_rksc_cgee']))     if 'lat_rksc_cgee'     in r else float('nan')

        kv_gain   = lat_b / _lat_kv_only    if _lat_kv_only    > 0 else float('nan')
        rksc_gain = lat_b / _lat_rksc_cgee  if _lat_rksc_cgee  > 0 else float('nan')

        kv_rr  = r.get('kv_reuse_rate', 0.0)
        n_q    = r.get('model_n_q_heads', '?')
        n_kv   = r.get('model_n_kv_heads', '?')
        n_l    = r.get('model_n_layers', '?')
        arch   = f'{n_l}L/{n_q}Q/{n_kv}K'

        _cgee_skip = r.get('cgee_skip_rate', float('nan'))
        _cgee_n    = r.get('cgee_call_n', 0)
        if _cgee_skip == _cgee_skip and _cgee_n > 0:
            _ci = 1.96 * (_cgee_skip * (1 - _cgee_skip) / max(_cgee_n, 1)) ** 0.5
            cgee_skip_s = f'{_cgee_skip:.0%}±{_ci:.0%}'
        else:
            cgee_skip_s = '—'

        def _ms(v): return f'{v:>7.0f}ms' if v == v else '     —'
        def _sp(v): return f'{v:>8.3f}×'  if v == v else '      —'

        print(f'  {model_short:<{_W}} {ds_name:<13}'
              f' {_ms(lat_b)} {_ms(_lat_kv_only)} {_ms(float("nan"))} {_ms(_lat_rksc_cgee)}'
              f' {_sp(kv_gain)} {_sp(rksc_gain)}'
              f' {cgee_skip_s:>11} {arch:>14}')

        rows_ext.append({
            'Model': mid, 'Dataset': ds_name,
            'Lat_batch_ms':    lat_b,
            'Lat_kv_ms':       _lat_kv_only,
            'Lat_rksc_ms':     _lat_rksc_cgee,
            'KV_gain':         kv_gain,
            'RKSC_gain':       rksc_gain,
            'KV_reuse':        kv_rr,
            'cgee_skip_rate':  _cgee_skip,
            'cgee_call_n':     _cgee_n,
            'cgee_saved_ms':   r.get('cgee_saved_ms', float('nan')),
            'cgee_speedup_vs_verify': r.get('cgee_speedup_vs_verify', float('nan')),
            'n_layers': n_l, 'n_kv_heads': n_kv, 'n_q_heads': n_q,
            'mem_saved_mb':    r.get('mem_saved_mb', float('nan')),
            'has_ground_truth':    r.get('has_ground_truth', False),
            'cgee_agreement_rate': r.get('cgee_agreement_rate', float('nan')),
            'cgee_accuracy_gt':    r.get('cgee_accuracy_gt',    float('nan')),
            'verify_accuracy_gt':  r.get('verify_accuracy_gt',  float('nan')),
            'accuracy_delta':      r.get('accuracy_delta',       float('nan')),
            'cgee_n_caused_errors':r.get('cgee_n_caused_errors', 0),
        })

if not rows_ext:
    print('No results yet — run the extended benchmark cell above.')



EXTENDED EVALUATION: Multi-Model, Multi-Dataset
Reference: Batched-no-KV  |  tok=8  prefix≈1024 tok  B=8
Speedup = mean(Batch-noKV) / mean(condition)  [ratio-of-means]
  Model                               Dataset        Batch-noKV        KV      CGEE      RKSC   KV gain  RKSC gain  CGEE skip%           Arch
  ----------------------------------------------------------------------------------------------------------------------------------


NameError: name 'extended_results' is not defined

In [ ]:
if rows_ext:
    df_ext = pd.DataFrame(rows_ext)

    print()
    print('── SCALING ANALYSIS: KV gain vs architecture ──')
    print(f'  {"Model":<35} {"KV gain":>9} {"RKSC gain":>10} {"KV saved":>9} {"Arch":>16}  MemΔ')
    print('  ' + '-'*90)
    by_model = (df_ext.groupby('Model')
                .agg({'KV_gain':'mean', 'RKSC_gain':'mean', 'Lat_batch_ms':'mean',
                      'Lat_kv_ms':'mean', 'n_layers':'first',
                      'n_kv_heads':'first', 'n_q_heads':'first', 'mem_saved_mb':'mean'})
                .sort_values('RKSC_gain', ascending=False))
    for mid, row in by_model.iterrows():
        arch = f'{row.n_layers}L/{row.n_q_heads}Q/{row.n_kv_heads}K'
        saved_ms = row.Lat_batch_ms - row.Lat_kv_ms
        mem_s = f'{row.mem_saved_mb:.0f} MB' if row.mem_saved_mb == row.mem_saved_mb else '—'
        print(f'  {mid.split("/")[-1]:<35} {row.KV_gain:>8.3f}× {row.RKSC_gain:>9.3f}×'
              f' {saved_ms:>7.0f} ms {arch:>16}  {mem_s}')

    _skip_data = [(r['cgee_skip_rate'], r['cgee_call_n'])
                  for r in rows_ext if r['cgee_skip_rate'] == r['cgee_skip_rate']]
    if _skip_data:
        _avg_skip = np.nanmean([x[0] for x in _skip_data])
        _avg_n    = np.mean([x[1] for x in _skip_data])
        _ci       = 1.96 * (_avg_skip * (1 - _avg_skip) / max(_avg_n, 1)) ** 0.5
        print()
        print(f'── CGEE skip-rate reliability ──')
        print(f'  Mean skip rate : {_avg_skip:.0%}  (N={_avg_n:.0f} calls/model, 95% CI ±{_ci:.0%})')

    print()
    print('── CGEE ACCURACY IMPACT ──')
    print(f'  {"Model":<35} {"Dataset":<13} {"Attempts":>9} {"Skipped":>8} {"Skip%":>6}'
          f' {"Agree%":>8} {"CGEE acc":>9} {"Verify acc":>11} {"Δacc":>6} {"Errors":>7}')
    print('  ' + '-'*110)
    for row in rows_ext:
        mn  = row['Model'].split('/')[-1]
        ds  = row['Dataset']
        cn  = row['cgee_call_n']
        sr  = row['cgee_skip_rate']
        sn  = int(round(sr * cn)) if sr == sr else 0
        ag  = row['cgee_agreement_rate']
        ca  = row['cgee_accuracy_gt']
        va  = row['verify_accuracy_gt']
        da  = row['accuracy_delta']
        er  = row['cgee_n_caused_errors']
        hgt = row['has_ground_truth']
        print(f'  {mn:<35} {ds:<13} {cn:>9} {sn:>8}'
              f' {sr:.0%} ' if sr == sr else f'  {"—":>6} ',
              end='')
        print(f'{ag:.1%} ' if ag == ag else f'   {"—":>6} ', end='')
        print(f'{ca:.1%} ' if (hgt and ca == ca) else f'  {"no GT":>6} ', end='')
        print(f'{va:.1%} ' if (hgt and va == va) else f'  {"no GT":>8} ', end='')
        print(f'{da:+.1%}' if (hgt and da == da) else f'  {"—":>5}', end='')
        print(f' {er:>7}' if hgt else f'   {"—":>5}')

    _all_errors  = sum(r['cgee_n_caused_errors'] for r in rows_ext if r['has_ground_truth'])
    _mean_agree  = np.nanmean([r['cgee_agreement_rate'] for r in rows_ext])
    _gt_rows     = [r for r in rows_ext if r['has_ground_truth']]
    _mean_delta  = np.nanmean([r['accuracy_delta'] for r in _gt_rows]) if _gt_rows else float('nan')
    print()
    print(f'  CGEE-induced errors (GT available): {_all_errors}')
    print(f'  Mean CGEE/verify agreement        : {_mean_agree:.1%}')
    if _gt_rows:
        print(f'  Mean accuracy delta               : {_mean_delta:+.1%}')
        if _all_errors == 0:
            print(f'  ✓ CGEE caused no accuracy regressions')
        else:
            print(f'  ⚠  {_all_errors} regressions — consider recalibrating cgee_gen_conf_threshold')
